# 06 · Google Maps API for reusable geographic review
Complementary tool: harmonization is still at 65 %; there is no fitted model.
It takes detections and a reference, opens satellite Google Maps and lets you filter,
record a human review per point and export observations.
Included case: seven VIIRS detections near Dos de Mayo, Curimaná, 2019.
It does not automatically confirm the fire or the historical date of the imagery.

## Setup and API key
Run in Jupyter or VS Code locally. Dependencies: pandas, pyproj, python-dotenv and
IPython. If missing: `%pip install pandas pyproj python-dotenv`.
Without a key, the data and server are tested; the map stays pending, with no calls to Google.

1. Create/select a Google Cloud project and enable the Maps JavaScript API.
2. Set up billing for a standard key and create an API key.
3. Restrict that key to the Maps JavaScript API and to these local sites:
   `http://127.0.0.1:8765/*` and `http://localhost:8765/*`.
4. In `.env` at the local root of the repo: `GOOGLE_MAPS_API_KEY=your_key`.
   That name is already excluded by .gitignore. An environment variable is also accepted.
5. Optional: `GOOGLE_MAPS_MAP_ID=your_map_id`; DEMO_MAP_ID is used in development.
6. Run the start cell; open the local link if the iframe does not show.

The key is not saved in outputs or HTML files. It is accessible to the browser by
design of the Maps JavaScript API; the real protection is the site/API restrictions.
Google also offers demo keys for prototypes; check their terms.

Primary sources:
https://developers.google.com/maps/documentation/javascript/get-api-key
https://developers.google.com/maps/documentation/javascript/advanced-markers/add-marker
https://developers.google.com/maps/api-security-best-practices

## Data contract
Normalized or FIRMS CSV files. WGS84 coordinates, unique identifiers, dates with
time zone and non-negative FRP. WGS84 ellipsoidal distances with pyproj.Geod.
Maximum 2,000 detections per case: filter region and period first.

In [1]:
from pathlib import Path

In [2]:
from datetime import datetime

In [3]:
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

In [4]:
import copy

In [5]:
import hashlib

In [6]:
import json

In [7]:
import math

In [8]:
import os

In [9]:
import re

In [10]:
import threading

In [11]:
from urllib.parse import urlsplit

In [12]:
import pandas as pd

In [13]:
from pyproj import Geod

Reusable implementation: `aware_utc`.

In [14]:
def aware_utc(value, field='fecha'):
    if value is None or isinstance(value, (bool, int, float)) or pd.isna(value):
        raise ValueError(f'{field}: se requiere una fecha con zona horaria.')
    try:
        stamp = pd.Timestamp(value)
    except Exception as exc:
        raise ValueError(f'{field}: fecha inválida.') from exc
    if pd.isna(stamp) or stamp.tzinfo is None:
        raise ValueError(f'{field}: incluye Z o un desplazamiento como -05:00.')
    return stamp.tz_convert('UTC').isoformat()

Reusable implementation: `numeric`.

In [15]:
def numeric(value, name, minimum, maximum):
    if value is None or isinstance(value, bool) or str(value).strip() == '':
        raise ValueError(f'{name}: valor numérico requerido.')
    try:
        number = float(value)
    except (ValueError, TypeError) as exc:
        raise ValueError(f'{name}: valor numérico inválido.') from exc
    if not math.isfinite(number) or not minimum <= number <= maximum:
        raise ValueError(f'{name}: valor fuera de rango [{minimum}, {maximum}].')
    return number

Reusable implementation: `find_column`.

In [16]:
def find_column(frame, aliases, required=True):
    for name in aliases:
        if name in frame.columns:
            return name
    if required:
        raise ValueError('Falta una columna: ' + ' / '.join(aliases))
    return None

Reusable implementation: `observation_times`.

In [17]:
def observation_times(frame):
    column = find_column(frame, ['timestamp_utc', 'acquired_at', 'timestamp', 'datetime'], False)
    if column:
        return [aware_utc(v, f'fila {i+1}') for i, v in enumerate(frame[column])]
    if not {'acq_date', 'acq_time'}.issubset(frame.columns):
        raise ValueError('Incluye timestamp_utc con zona, o acq_date y acq_time FIRMS (UTC).')
    stamps = []
    for i, row in frame.iterrows():
        clock = str(row.acq_time).strip()
        if not re.fullmatch(r'\d{1,4}', clock):
            raise ValueError(f'Fila {i}: acq_time debe tener formato HHMM.')
        try:
            dt = datetime.strptime(str(row.acq_date)+' '+clock.zfill(4), '%Y-%m-%d %H%M')
        except ValueError as exc:
            raise ValueError(f'Fila {i}: fecha/hora FIRMS inválida.') from exc
        stamps.append(pd.Timestamp(dt, tz='UTC').isoformat())
    return stamps

Reusable implementation: `text_or_none`.

In [18]:
def text_or_none(value):
    return None if value is None or pd.isna(value) else str(value)

Reusable implementation: `prepare_case`.

In [19]:
def prepare_case(frame, *, name, case_id, reference_latitude, reference_longitude,
                 event_start, event_end, radius_m=2000, reference_label='Referencia del analista',
                 reference_kind='analyst_reference', sources=None):
    """Admite CSV normalizados o FIRMS; no altera el DataFrame recibido."""
    if not isinstance(frame, pd.DataFrame) or not 1 <= len(frame) <= 2000:
        raise ValueError('Usa un DataFrame con entre 1 y 2.000 detecciones por caso.')
    ref_lat = numeric(reference_latitude, 'latitud de referencia', -90, 90)
    ref_lon = numeric(reference_longitude, 'longitud de referencia', -180, 180)
    radius = numeric(radius_m, 'radio en metros', 50, 500000)
    start, end = aware_utc(event_start, 'inicio'), aware_utc(event_end, 'final')
    if pd.Timestamp(start) > pd.Timestamp(end):
        raise ValueError('El inicio debe ser anterior al final.')
    lat = find_column(frame, ['latitude_deg', 'latitude', 'lat'])
    lon = find_column(frame, ['longitude_deg', 'longitude', 'lon', 'lng'])
    times = observation_times(frame)
    detections = []
    geod = Geod(ellps='WGS84')
    for n, (_, row) in enumerate(frame.iterrows(), 1):
        a = numeric(row[lat], f'latitud fila {n}', -90, 90)
        b = numeric(row[lon], f'longitud fila {n}', -180, 180)
        platform = text_or_none(row.get('platform', row.get('satellite'))) or 'Sin plataforma'
        raw_sensor = text_or_none(row.get('instrument_normalized', row.get('instrument'))) or ''
        instrument = 'VIIRS' if raw_sensor.upper() in {'VIIRS', 'SNPP'} or platform.upper() in {'SNPP', 'SUOMI-NPP', 'N20', 'N21', 'NOAA-20', 'NOAA-21'} else ('MODIS' if raw_sensor.upper() == 'MODIS' or platform in {'Aqua', 'Terra'} else raw_sensor or 'Otro')
        raw_frp = row.get('frp_mw', row.get('frp'))
        frp = None if raw_frp is None or pd.isna(raw_frp) else numeric(raw_frp, 'FRP', 0, 1e12)
        _, _, distance = geod.inv(ref_lon, ref_lat, b, a)
        detections.append({'id': text_or_none(row.get('id')) or f'P{n:03}',
                           'latitude': a, 'longitude': b, 'timestamp_utc': times[n-1],
                           'platform': platform, 'instrument': instrument, 'frp_mw': frp,
                           'confidence': text_or_none(row.get('confidence')), 'distance_m': float(distance),
                           'source_file': text_or_none(row.get('source_member')),
                           'source_row': text_or_none(row.get('source_row'))})
    if len({p['id'] for p in detections}) != len(detections):
        raise ValueError('Los identificadores de detección deben ser únicos.')
    data_hash = hashlib.sha256(json.dumps(detections, sort_keys=True, allow_nan=False).encode()).hexdigest()
    result = {'schema_version': 1, 'id': str(case_id), 'name': str(name),
              'reference': {'latitude': ref_lat, 'longitude': ref_lon, 'label': str(reference_label),
                            'kind': str(reference_kind), 'fire_position_error_m': None},
              'interval': {'start': start, 'end': end}, 'radius_m': radius,
              'sources': sources or [], 'data_sha256': data_hash, 'detections': detections,
              'incident_attribution_confirmed': False}
    json.dumps(result, allow_nan=False)
    return result

Reusable implementation: `load_csv`.

In [20]:
def load_csv(path):
    """Mantiene ceros iniciales y valores originales como texto al leer FIRMS."""
    return pd.read_csv(path, dtype='string')

Reusable implementation: `google_settings`.

In [21]:
def google_settings(repo_root=None):
    """La clave nunca se imprime ni se inserta en las salidas del notebook."""
    env_file = Path(repo_root or Path.cwd()) / '.env'
    file_values = {}
    if env_file.exists():
        from dotenv import dotenv_values
        file_values = dotenv_values(env_file)
    key = os.environ.get('GOOGLE_MAPS_API_KEY') or file_values.get('GOOGLE_MAPS_API_KEY') or ''
    map_id = os.environ.get('GOOGLE_MAPS_MAP_ID') or file_values.get('GOOGLE_MAPS_MAP_ID') or 'DEMO_MAP_ID'
    return {'api_key': key.strip(), 'map_id': map_id.strip()}

Reusable implementation: `MapsSession`.

In [22]:
class MapsSession:
    """Servidor local del mapa; vive mientras el kernel del notebook esté activo."""
    def __init__(self, case, html, *, api_key='', map_id='DEMO_MAP_ID', port=8765):
        self.case = copy.deepcopy(case)
        self._key = api_key
        self._map_id = map_id
        self._html = html.encode('utf-8')
        self._reviews = {}
        self._lock = threading.Lock()
        self._server = ThreadingHTTPServer(('127.0.0.1', port), self._handler())
        self.port = self._server.server_port
        self.url = f'http://127.0.0.1:{self.port}/'
        self._thread = threading.Thread(target=self._server.serve_forever, daemon=True)
        self._thread.start()

    def __repr__(self):
        return f'MapsSession(url={self.url!r}, configured={bool(self._key)})'

    @property
    def reviews(self):
        with self._lock:
            return copy.deepcopy(self._reviews)

    def close(self):
        if self._thread.is_alive():
            self._server.shutdown()
        self._server.server_close()

    def embed(self, height=790):
        from IPython.display import IFrame, display
        display(IFrame(self.url, width='100%', height=height))

    def export_reviews(self, path):
        result = {'schema_version': 1, 'case_id': self.case['id'],
                  'data_sha256': self.case['data_sha256'],
                  'review_type': 'human_geographic_review_not_incident_confirmation',
                  'observations': self.reviews}
        Path(path).write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
        return Path(path)

    def _validate_review(self, item):
        if not isinstance(item, dict) or item.get('detection_id') not in {p['id'] for p in self.case['detections']}:
            raise ValueError('Detección desconocida.')
        if item.get('decision') not in {'pending', 'spatially_consistent', 'needs_review'}:
            raise ValueError('Estado de revisión inválido.')
        if item['decision'] != 'pending' and not self._key:
            raise ValueError('Configura Google Maps antes de registrar una revisión geográfica.')
        notes = item.get('notes', '')
        if not isinstance(notes, str) or len(notes) > 2000:
            raise ValueError('Observación demasiado larga.')
        image_date = item.get('imagery_date')
        if image_date:
            if not isinstance(image_date, str) or not re.fullmatch(r'\d{4}-\d{2}-\d{2}', image_date):
                raise ValueError('Fecha de imagen inválida.')
            datetime.strptime(image_date, '%Y-%m-%d')
        return {'decision': item['decision'], 'imagery_date': image_date or None, 'notes': notes,
                'reviewed_at': aware_utc(item.get('reviewed_at'), 'fecha de revisión')}

    def _handler(self):
        session = self

        class Handler(BaseHTTPRequestHandler):
            def log_message(self, *args):
                pass

            def _allowed_host(self):
                return self.headers.get('Host') in {f'127.0.0.1:{session.port}', f'localhost:{session.port}'}

            def respond(self, status, content, content_type='application/json; charset=utf-8'):
                payload = content if isinstance(content, bytes) else json.dumps(content, ensure_ascii=False).encode()
                self.send_response(status)
                self.send_header('Content-Type', content_type)
                self.send_header('Content-Length', str(len(payload)))
                self.send_header('Cache-Control', 'no-store')
                self.send_header('X-Content-Type-Options', 'nosniff')
                self.send_header('Referrer-Policy', 'strict-origin-when-cross-origin')
                self.end_headers()
                self.wfile.write(payload)

            def do_GET(self):
                if not self._allowed_host():
                    return self.respond(403, {'error': 'Host no autorizado.'})
                route = urlsplit(self.path).path
                if route == '/':
                    return self.respond(200, session._html, 'text/html; charset=utf-8')
                if route == '/config':
                    return self.respond(200, {'configured': bool(session._key), 'apiKey': session._key, 'mapId': session._map_id})
                if route == '/data':
                    return self.respond(200, session.case)
                if route == '/health':
                    return self.respond(200, {'ok': True, 'configured': bool(session._key), 'detections': len(session.case['detections'])})
                return self.respond(404, {'error': 'Recurso no disponible.'})

            def do_POST(self):
                if not self._allowed_host():
                    return self.respond(403, {'error': 'Host no autorizado.'})
                if urlsplit(self.path).path != '/reviews':
                    return self.respond(404, {'error': 'Recurso no disponible.'})
                try:
                    size = int(self.headers.get('Content-Length', 0))
                    if not 2 <= size <= 12000:
                        raise ValueError('Tamaño de solicitud inválido.')
                    item = json.loads(self.rfile.read(size))
                    review = session._validate_review(item)
                    with session._lock:
                        session._reviews[item['detection_id']] = review
                except (ValueError, TypeError, KeyError, UnicodeError):
                    return self.respond(400, {'error': 'Revisión inválida; verifica sus campos.'})
                return self.respond(200, {'saved': True})

        return Handler

Viewer template; it contains no credentials.

In [23]:
MAP_HTML = r"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Revisión geográfica de detecciones</title>
<style>
:root{font-family:Arial,"Segoe UI",sans-serif;color:#24394b;background:#fff;font-size:14px;--blue:#155fa0;--muted:#516675;--line:#d7e0e6}
*{box-sizing:border-box}body{margin:0}button,input,select,textarea{font:inherit}button{cursor:pointer;border:1px solid #adbdc9;border-radius:5px;background:#fff;color:#24394b;padding:8px 12px}button:hover{background:#eef4f8}button:disabled{cursor:default;opacity:.5}button.primary{background:var(--blue);color:#fff;border-color:var(--blue)}:focus-visible{outline:3px solid #2b7ec0;outline-offset:2px}
header{padding:16px 20px;border-bottom:1px solid var(--line);display:flex;gap:16px;align-items:center;justify-content:space-between}h1{font-size:20px;margin:0 0 5px}h2{font-size:16px;margin:0 0 12px}p{line-height:1.5;margin:6px 0}.muted{color:var(--muted);font-size:12px}.workspace{display:grid;grid-template-columns:290px 1fr;min-height:600px}aside{padding:18px;border-right:1px solid var(--line);overflow:auto;max-height:740px}label{display:block;font-size:12px;font-weight:600;margin:12px 0 6px}input,select,textarea{width:100%;padding:8px;border:1px solid #a8b8c5;border-radius:4px;background:#fff;color:#24394b}textarea{resize:vertical;min-height:66px}.time-label{font-size:11px;color:var(--muted)}.point-list{list-style:none;margin:16px 0;padding:0;display:grid;gap:4px}.point-list button{width:100%;text-align:left;display:flex;align-items:center;gap:10px;padding:9px}.point-list button[aria-pressed=true]{background:#eaf3fb;border-color:var(--blue)}.point-id{display:grid;place-items:center;min-width:29px;height:29px;border-radius:50%;background:#a94e0a;color:#fff;font-weight:bold;font-size:11px}.point-id.modis{background:#155fa0}.point-info{flex:1}.point-info strong{display:block;font-size:13px}.dot{width:8px;height:8px;border-radius:50%;background:#718390}.dot.done{background:#14744f}.map-pane{position:relative;min-width:0;background:#edf2f5;min-height:600px}.map{position:absolute;inset:0}.map-message{position:absolute;inset:0;display:grid;place-items:center;padding:28px;background:#edf2f5;z-index:2}.setup{background:white;border:1px solid var(--line);border-radius:8px;max-width:520px;padding:24px}.setup h2{font-size:21px;line-height:1.3}.setup pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#f1f5f8;padding:12px;font-size:12px}.setup a{color:var(--blue)}.legend{position:absolute;left:12px;bottom:34px;background:#fff;padding:9px 12px;border-radius:5px;z-index:1;max-width:calc(100% - 24px);font-size:12px}.status{padding:9px 18px;background:#f5f8fa;border-bottom:1px solid var(--line);font-size:12px;line-height:1.4}.detail{border-top:1px solid var(--line);padding-top:15px;margin-top:15px}.detail dl{display:grid;grid-template-columns:1fr 1.2fr;gap:7px;font-size:12px}.detail dt{color:var(--muted)}.detail dd{margin:0;overflow-wrap:anywhere}.feedback{font-size:12px;min-height:20px;color:#155f41}.actions{display:flex;gap:7px;margin-top:12px;flex-wrap:wrap}.map-pin{background:#a94e0a;color:white;min-width:31px;height:31px;border:2px solid white;border-radius:50%;display:grid;place-items:center;font-weight:700;box-shadow:0 2px 5px #0003}.map-pin.modis{background:#155fa0}.map-pin.selected{outline:3px solid #24394b}.map-pin.reference{border-radius:4px;background:#24394b;min-width:25px;height:25px}.hidden{display:none!important}.form-error{color:#9b2626}
@media(max-width:650px){header{padding:12px;align-items:flex-start;flex-wrap:wrap}.workspace{grid-template-columns:1fr}.map-pane{grid-row:1;min-height:380px}aside{border-right:0;border-top:1px solid var(--line);max-height:none}.point-list{grid-template-columns:repeat(2,minmax(0,1fr))}.setup{padding:17px}.setup h2{font-size:18px}.map-message{padding:14px}.status{padding:9px 12px}h1{font-size:18px}}
</style>
</head>
<body>
<header><div><h1 id="title">Revisión geográfica</h1><p class="muted" id="subtitle">Preparando detecciones…</p></div><button id="export">Exportar revisión JSON</button></header>
<div class="status" id="status" role="status" aria-live="polite">Cargando configuración local…</div>
<main class="workspace">
<aside>
<h2>Explorar detecciones</h2>
<label for="sensor">Plataforma</label><select id="sensor"><option value="all">Todas</option></select>
<label for="start">Desde <span class="time-label">UTC</span></label><input id="start" type="datetime-local" step="1">
<label for="end">Hasta <span class="time-label">UTC</span></label><input id="end" type="datetime-local" step="1">
<label for="radius">Radio desde la referencia (km)</label><input id="radius" type="number" min="0.05" max="500" step="0.1" value="2">
<div class="actions"><button id="fit" disabled>Centrar puntos</button><button id="reset">Restablecer filtros</button></div>
<p class="muted" id="counts"></p><p class="form-error muted" id="filterError" role="alert"></p>
<ul class="point-list" id="points" aria-label="Detecciones filtradas"></ul>
<section class="detail hidden" id="detail" aria-label="Revisar detección">
<h2 id="pointTitle"></h2><dl id="facts"></dl>
<label for="decision">Revisión geográfica</label>
<select id="decision" disabled><option value="pending">Sin revisar</option><option value="spatially_consistent">Ubicación coherente</option><option value="needs_review">Requiere revisión</option></select>
<label for="imageryDate">Fecha de la imagen observada</label><input id="imageryDate" type="date"><p class="muted">Déjala vacía si Google Maps no informa la fecha. No implica imagen de 2019.</p>
<label for="notes">Observaciones y evidencia</label><textarea id="notes" maxlength="2000" placeholder="Describe qué observaste y qué falta comprobar."></textarea>
<div class="actions"><button id="save" class="primary">Guardar observación</button></div><p id="feedback" class="feedback" aria-live="polite"></p>
</section>
</aside>
<section class="map-pane" aria-label="Google Maps">
<div id="map" class="map"></div>
<div id="mapMessage" class="map-message"><div class="setup"><h2 id="mapHeading">Preparando Google Maps</h2><p id="mapExplanation">Los puntos están listos para cargarse.</p><details id="setupDetails"><summary>Cómo configurar la clave</summary><p>En Google Cloud, habilita Maps JavaScript API y configura facturación para una clave estándar. Restringe la clave a esta API y a estas direcciones locales:</p><pre>http://127.0.0.1:8765/*
http://localhost:8765/*</pre><p>En tu archivo .env local:</p><pre>GOOGLE_MAPS_API_KEY=tu_clave
GOOGLE_MAPS_MAP_ID=DEMO_MAP_ID</pre><p>Después vuelve a ejecutar la celda que inicia el mapa. DEMO_MAP_ID sirve para desarrollo.</p><a href="https://developers.google.com/maps/documentation/javascript/get-api-key" target="_blank" rel="noopener">Configuración oficial de Google Maps</a></details></div></div>
<div class="legend" id="legend">Los puntos son centros de píxeles. La referencia de localidad no es un perímetro de incendio.</div>
</section>
</main>
<script>
'use strict';
let caseData, config, map, circle, info, referenceMarker, selectedId, mapReady=false;
const markers=new Map(), reviews=new Map();
const $=id=>document.getElementById(id);
const localTime=iso=>new Intl.DateTimeFormat('es-PE',{timeZone:'America/Lima',dateStyle:'short',timeStyle:'short'}).format(new Date(iso));
const timeValue=iso=>new Date(iso).toISOString().slice(0,19);
function showMessage(heading,description){$('mapMessage').classList.remove('hidden');$('mapHeading').textContent=heading;$('mapExplanation').textContent=description;}
function safeText(tag,text){const node=document.createElement(tag);node.textContent=text;return node;}
function setDefaults(){
  $('start').value=timeValue(caseData.interval.start);$('end').value=timeValue(caseData.interval.end);
  $('radius').value=caseData.radius_m/1000;$('sensor').value='all';
}
function filtered(){
  const radius=Number($('radius').value)*1000,from=Date.parse($('start').value+'Z'),to=Date.parse($('end').value+'Z');
  if(!Number.isFinite(radius)||radius<50||radius>500000||!Number.isFinite(from)||!Number.isFinite(to)||from>to){
    $('filterError').textContent='Revisa el radio y las fechas: el inicio debe ser anterior al final.';return [];
  }
  $('filterError').textContent='';
  return caseData.detections.filter(p=>p.distance_m<=radius&&Date.parse(p.timestamp_utc)>=from&&Date.parse(p.timestamp_utc)<=to&&($('sensor').value==='all'||p.platform===$('sensor').value));
}
function drawList(visible){
  $('points').replaceChildren();
  for(const p of visible){
    const li=document.createElement('li'),button=document.createElement('button');button.type='button';button.setAttribute('aria-pressed',String(p.id===selectedId));
    const badge=safeText('span',p.id);badge.className='point-id'+(p.instrument==='MODIS'?' modis':'');
    const content=document.createElement('span');content.className='point-info';content.append(safeText('strong',p.platform),safeText('span',localTime(p.timestamp_utc)+' · Perú'));
    const dot=document.createElement('span');dot.className='dot'+(reviews.get(p.id)?.decision&&reviews.get(p.id).decision!=='pending'?' done':'');
    button.append(badge,content,dot);button.addEventListener('click',()=>selectPoint(p.id));li.append(button);$('points').append(li);
  }
  if(!visible.length)$('points').append(safeText('li','No hay detecciones con estos filtros. No demuestra ausencia de fuego.'));
}
function render(){
  const visible=filtered(),ids=new Set(visible.map(p=>p.id));
  $('counts').textContent=visible.length+' de '+caseData.detections.length+' detecciones · '+[...reviews.values()].filter(r=>r.decision!=='pending').length+' revisadas';
  drawList(visible);
  for(const [id,marker] of markers)marker.map=ids.has(id)?map:null;
  if(circle)circle.setRadius(Number($('radius').value)*1000||caseData.radius_m);
}
function selectPoint(id){
  selectedId=id;const p=caseData.detections.find(p=>p.id===id),review=reviews.get(id)||{};
  $('detail').classList.remove('hidden');$('pointTitle').textContent=p.id+' · '+p.platform;
  $('facts').replaceChildren();
  for(const [name,value] of [['Coordenadas',p.latitude.toFixed(5)+', '+p.longitude.toFixed(5)],['Hora Perú',localTime(p.timestamp_utc)],['Distancia WGS84',(p.distance_m/1000).toFixed(3)+' km'],['Confianza',p.confidence??'No informada'],['FRP',p.frp_mw==null?'No informada':p.frp_mw+' MW']])$('facts').append(safeText('dt',name),safeText('dd',String(value)));
  $('decision').value=review.decision||'pending';$('imageryDate').value=review.imagery_date||'';$('notes').value=review.notes||'';$('feedback').textContent='';
  for(const [pid,marker]of markers)marker.firstChild?.classList.toggle('selected',pid===id);
  if(mapReady){map.panTo({lat:p.latitude,lng:p.longitude});const content=document.createElement('div');content.append(safeText('strong',p.id+' · '+p.platform),safeText('p',localTime(p.timestamp_utc)+' · hora Perú'));info.setContent(content);info.open({map,anchor:markers.get(id),shouldFocus:false});}
  drawList(filtered());
}
function fit(){
  const visible=filtered();if(!mapReady||!visible.length)return;
  const bounds=new google.maps.LatLngBounds();bounds.extend({lat:caseData.reference.latitude,lng:caseData.reference.longitude});
  visible.forEach(p=>bounds.extend({lat:p.latitude,lng:p.longitude}));map.fitBounds(bounds,65);
}
function exportReview(){
  const payload={schema_version:1,case_id:caseData.id,data_sha256:caseData.data_sha256,exported_at:new Date().toISOString(),
    review_type:'human_geographic_review_not_incident_confirmation',imagery_temporal_match_verified:false,
    reference:caseData.reference,interval:caseData.interval,filters:{radius_km:Number($('radius').value),start_utc:$('start').value+'Z',end_utc:$('end').value+'Z',platform:$('sensor').value},
    observations:caseData.detections.map(p=>({detection_id:p.id,latitude:p.latitude,longitude:p.longitude,timestamp_utc:p.timestamp_utc,...(reviews.get(p.id)||{decision:'pending',imagery_date:null,notes:''})}))};
  const url=URL.createObjectURL(new Blob([JSON.stringify(payload,null,2)],{type:'application/json'}));
  const a=document.createElement('a');a.href=url;a.download='revision_'+caseData.id+'.json';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);
}
async function saveReview(){
  if(!selectedId)return;
  const review={decision:$('decision').value,imagery_date:$('imageryDate').value||null,notes:$('notes').value,reviewed_at:new Date().toISOString()};
  if(review.decision!=='pending'&&!mapReady){$('feedback').textContent='Activa Google Maps antes de registrar una revisión geográfica.';return;}
  try{const response=await fetch('/reviews',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({detection_id:selectedId,...review})});
    if(!response.ok)throw Error('No se pudo guardar.');reviews.set(selectedId,review);render();$('feedback').textContent='Observación guardada en esta sesión. Exporta el JSON para conservarla.';
  }catch(error){$('feedback').textContent=error.message;}
}
async function initializeMap(){
  try{
    const {Map:GoogleMap}=await google.maps.importLibrary('maps');const {AdvancedMarkerElement}=await google.maps.importLibrary('marker');
    map=new GoogleMap($('map'),{center:{lat:caseData.reference.latitude,lng:caseData.reference.longitude},zoom:14,mapId:config.mapId,mapTypeId:'satellite',mapTypeControl:true,streetViewControl:false,fullscreenControl:true,gestureHandling:'cooperative'});
    info=new google.maps.InfoWindow();
    circle=new google.maps.Circle({map,center:{lat:caseData.reference.latitude,lng:caseData.reference.longitude},radius:caseData.radius_m,strokeColor:'#24394b',strokeOpacity:.8,strokeWeight:2,fillColor:'#155fa0',fillOpacity:.04,clickable:false});
    referenceMarker=new AdvancedMarkerElement({map,position:{lat:caseData.reference.latitude,lng:caseData.reference.longitude},title:'Referencia de localidad; no ubicación exacta del fuego'});const ref=safeText('span','R');ref.className='map-pin reference';referenceMarker.append(ref);
    for(const p of caseData.detections){const marker=new AdvancedMarkerElement({map,position:{lat:p.latitude,lng:p.longitude},title:p.id+' · '+p.platform+' · '+localTime(p.timestamp_utc),gmpClickable:true});const pin=safeText('span',p.id);pin.className='map-pin'+(p.instrument==='MODIS'?' modis':'');marker.append(pin);marker.addListener('click',()=>selectPoint(p.id));markers.set(p.id,marker);}
    mapReady=true;$('mapMessage').classList.add('hidden');$('decision').disabled=false;$('fit').disabled=false;
    $('status').textContent='Google Maps activo. Revisión geográfica; fecha histórica de la imagen sin verificar.';render();fit();
  }catch(error){mapReady=false;showMessage('No se pudo iniciar Google Maps','Revisa la clave, Maps JavaScript API, la facturación y las restricciones de dominio.');$('status').textContent='Mapa no disponible; revisión geográfica pendiente.';}
}
function loadGoogle(){
  if(!config.configured){showMessage('Falta configurar la API key','Las '+caseData.detections.length+' detecciones están cargadas. El mapa satelital se activará al configurar Google Maps.');$('status').textContent='Datos listos · mapa pendiente de API key · sin llamadas a Google Maps';return;}
  $('setupDetails').open=false;window.mapsValidationReady=initializeMap;
  window.gm_authFailure=()=>{mapReady=false;$('decision').disabled=true;$('fit').disabled=true;showMessage('Google rechazó la configuración','Revisa las restricciones HTTP y de API, y la facturación del proyecto.');$('status').textContent='Error de autenticación de Google Maps.';};
  const params=new URLSearchParams({key:config.apiKey,callback:'mapsValidationReady',loading:'async',v:'weekly',libraries:'maps,marker',language:'es'});
  const script=document.createElement('script');script.async=true;script.src='https://maps.googleapis.com/maps/api/js?'+params;
  script.onerror=()=>showMessage('No se pudo conectar con Google Maps','Revisa la conexión de red y vuelve a cargar el visor.');document.head.append(script);
}
async function boot(){
  try{
    const responses=await Promise.all([fetch('/data'),fetch('/config')]);if(responses.some(r=>!r.ok))throw Error('No se pudo leer la sesión local.');
    [caseData,config]=await Promise.all(responses.map(r=>r.json()));$('title').textContent=caseData.name;$('subtitle').textContent=caseData.detections.length+' detecciones · '+caseData.reference.label;
    for(const platform of [...new Set(caseData.detections.map(p=>p.platform))]){const option=safeText('option',platform);option.value=platform;$('sensor').append(option);}
    for(const id of ['sensor','start','end','radius'])$(id).addEventListener('change',render);
    $('fit').addEventListener('click',fit);$('reset').addEventListener('click',()=>{setDefaults();render();fit();});$('save').addEventListener('click',saveReview);$('export').addEventListener('click',exportReview);
    setDefaults();render();if(caseData.detections.length)selectPoint(caseData.detections[0].id);loadGoogle();
  }catch(error){showMessage('Sesión no disponible',error.message);$('status').textContent='Vuelve a ejecutar la celda de inicio del notebook.';}
}
boot();
</script>
</body></html>
"""

## Prepare the case
Change the DataFrame and the prepare_case parameters to reuse it.
Another CSV: `detections = load_csv("path/to/file.csv")`.
latitude/longitude or latitude_deg/longitude_deg are accepted, and timestamp_utc with
time zone, or FIRMS acq_date/acq_time in UTC. platform/satellite and confidence are optional.
The ANA reference is for the locality, not the exact point or perimeter of the fire.

In [24]:
SAMPLE_POINTS = [{'latitude_deg': -8.45571, 'longitude_deg': -75.14859, 'timestamp_utc': '2019-09-01 05:49:00+00:00', 'platform': 'Suomi-NPP', 'confidence': 'n', 'frp_mw': 6.08, 'source_member': 'fire_archive_SV-C2_848901.csv', 'source_row': 525748, 'id': 'V01'}, {'latitude_deg': -8.45358, 'longitude_deg': -75.14725, 'timestamp_utc': '2019-09-01 05:49:00+00:00', 'platform': 'Suomi-NPP', 'confidence': 'n', 'frp_mw': 3.23, 'source_member': 'fire_archive_SV-C2_848901.csv', 'source_row': 525762, 'id': 'V02'}, {'latitude_deg': -8.45844, 'longitude_deg': -75.14808, 'timestamp_utc': '2019-09-01 05:49:00+00:00', 'platform': 'Suomi-NPP', 'confidence': 'n', 'frp_mw': 3.23, 'source_member': 'fire_archive_SV-C2_848901.csv', 'source_row': 525746, 'id': 'V03'}, {'latitude_deg': -8.45278, 'longitude_deg': -75.14744, 'timestamp_utc': '2019-09-01 18:26:00+00:00', 'platform': 'Suomi-NPP', 'confidence': 'n', 'frp_mw': 4.99, 'source_member': 'fire_archive_SV-C2_848901.csv', 'source_row': 526575, 'id': 'V04'}, {'latitude_deg': -8.45616, 'longitude_deg': -75.14692, 'timestamp_utc': '2019-09-01 18:26:00+00:00', 'platform': 'Suomi-NPP', 'confidence': 'n', 'frp_mw': 4.99, 'source_member': 'fire_archive_SV-C2_848901.csv', 'source_row': 526578, 'id': 'V05'}, {'latitude_deg': -8.44087, 'longitude_deg': -75.16026, 'timestamp_utc': '2019-09-01 18:26:00+00:00', 'platform': 'Suomi-NPP', 'confidence': 'n', 'frp_mw': 4.81, 'source_member': 'fire_archive_SV-C2_848901.csv', 'source_row': 526569, 'id': 'V06'}, {'latitude_deg': -8.45562, 'longitude_deg': -75.14333, 'timestamp_utc': '2019-09-01 18:26:00+00:00', 'platform': 'Suomi-NPP', 'confidence': 'n', 'frp_mw': 3.94, 'source_member': 'fire_archive_SV-C2_848901.csv', 'source_row': 526579, 'id': 'V07'}]
detections = pd.DataFrame(SAMPLE_POINTS)
case = prepare_case(
    detections, name='Curimaná 2019 · revisión geográfica', case_id='curimana_2019',
    reference_latitude=-8.453540639633092, reference_longitude=-75.15789724255865,
    reference_label='Dos de Mayo · referencia de localidad ANA', reference_kind='locality_proxy',
    event_start='2019-08-31T17:30:00-05:00', event_end='2019-09-02T02:00:00-05:00', radius_m=2000,
    sources=[
        'https://portal.indeci.gob.pe/wp-content/uploads/2019/09/REPORTE-COMPLEMENTARIO-N%C2%BA-2027-06SEP2019-INCENDIO-FORESTAL-EN-EL-DISTRITO-DE-CURIMANA-UCAYALI-02.pdf',
        'https://www.ana.gob.pe/sites/default/files/normatividad/files/001-RESOLUCI%C3%93N%20DIRECTORAL%20N%C2%B0%20103-2019-ANA-AAA.U.pdf'])
from IPython.display import display
display(pd.DataFrame(case['detections'])[['id','platform','latitude','longitude','timestamp_utc','distance_m']].round({'distance_m':1}))

,id,platform,latitude,longitude,timestamp_utc,distance_m
0,V01,Suomi-NPP,-8.45571,-75.14859,2019-09-01T05:49:00+00:00,1052.6
1,V02,Suomi-NPP,-8.45358,-75.14725,2019-09-01T05:49:00+00:00,1172.5
2,V03,Suomi-NPP,-8.45844,-75.14808,2019-09-01T05:49:00+00:00,1209.2
3,V04,Suomi-NPP,-8.45278,-75.14744,2019-09-01T18:26:00+00:00,1154.6
4,V05,Suomi-NPP,-8.45616,-75.14692,2019-09-01T18:26:00+00:00,1243.0
5,V06,Suomi-NPP,-8.44087,-75.16026,2019-09-01T18:26:00+00:00,1425.3
6,V07,Suomi-NPP,-8.45562,-75.14333,2019-09-01T18:26:00+00:00,1620.5


## Local checks before opening the map
These checks do not test a real call to Google. The satellite SDK is still
waiting for an API key. Geocoding, Places, Routes and Static Maps are not called.

In [25]:
assert len(case['detections']) == 7
assert all(1000 < p['distance_m'] < 2000 for p in case['detections'])
assert all(p['instrument'] == 'VIIRS' for p in case['detections'])
assert case['interval']['start'] == '2019-08-31T22:30:00+00:00'
try:
    aware_utc('2019-09-01 05:49')
    raise AssertionError('Se aceptó una hora sin zona.')
except ValueError:
    pass
print('Siete detecciones válidas; UTC y distancias WGS84 comprobados.')

Siete detecciones válidas; UTC y distancias WGS84 comprobados.


## Start the viewer
It uses AdvancedMarkerElement and a search circle. It lives while the kernel
is active. The GitHub preview does not run servers: run it locally.

In [26]:
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name in {'notebooks', 'validation'}:
    REPO_ROOT = REPO_ROOT.parent
settings = google_settings(REPO_ROOT)
if globals().get('maps_session'):
    maps_session.close()
maps_session = MapsSession(case, MAP_HTML, api_key=settings['api_key'], map_id=settings['map_id'], port=8765)
print('Visor local:', maps_session.url)
print('API key configurada:', bool(settings['api_key']))
maps_session.embed()

Visor local: http://127.0.0.1:8765/
API key configurada: False


## Human review and export
Select a point, record geographic consistency and notes. The image date
can be left empty: it is not interpreted as 2019. Those states do not confirm the fire.
Export the JSON before stopping the kernel; without exporting, the reviews are temporary.
You can also use the Export review JSON button inside the map.

In [27]:
display(pd.DataFrame.from_dict(maps_session.reviews, orient='index'))
# Después de revisar:
# maps_session.export_reviews(REPO_ROOT / 'revision_curimana.json')
# Al terminar:
# maps_session.close()

""


## Limits
The points are pixel centres. The circle is not burned area or a perimeter.
An image without a compatible date does not confirm a scar of the 2019 fire.
This viewer does not automatically get historical dates of the base map.
Absence of MODIS does not prove an omission without checking clouds/overpasses/coverage.
Geographic comparison and statistical validation of the harmonization are different things.
Local use: remote Colab/Jupyter needs a proxy/HTTPS and its own restrictions.
Google Maps with a standard key may incur charges; check the project's quotas.